In [1]:
# import sys, os
# sys.path.append(os.path.join('/home/module'))
# import pgd_imp_old as imp
# import pandas as pd

# import math
# import pandas as pd
# from google.cloud import bigquery
# from google.oauth2 import service_account



In [2]:
import math
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import bigquery
from google.oauth2 import service_account


# setup BigQuery client sekali
key_path = '/home/jupyterhub/SA/jupyter-pgd-dev-data-analytics-296ca984dd2f.json'
credentials = service_account.Credentials.from_service_account_file(
    key_path, scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
bq_client = bigquery.Client(credentials=credentials, project=credentials.project_id)


/opt/conda/lib/python3.8/site-packages/google/cloud/bigquery_storage_v1/__init__.py:57: FutureWarning: You are using a non-supported Python version (3.8.16).  Google will not post any further updates to google.cloud.bigquery_storage_v1 supporting this Python version. Please upgrade to the latest Python version, or at least to Python 3.9, and then update google.cloud.bigquery_storage_v1.
  warnings.warn(


In [4]:
dtype_to_bq = {
    'object': 'STRING',
    'int64': 'INT64',
    'int32': 'INT64',
    'float64': 'FLOAT64',
    'datetime64[ns]': 'TIMESTAMP',
}

In [5]:
destination_table = "pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_det_suspicious_cust"
summary_table = "pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_sum_suspicious_cust"
case_table = "pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_case_suspicious_cust"
emp_detail = "pgd-dev-data-analytics.datamart_audit.dm_tbl_cust_detail"
up_kontrak_det = "pgd-dev-data-analytics.datamart_audit.dm_grafik_sidik_up_kontrak_cust"
bj_det = "pgd-dev-data-analytics.datamart_audit.dm_grafik_sidik_bj_cust"

### Inisialisasi BigQuery untuk ambil tabel

In [6]:
# from google.cloud import bigquery
# client = bigquery.Client()


In [7]:
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta

today = datetime.today() - relativedelta(days=1)
yesterday = today - relativedelta(days=1)
last_week = today - relativedelta(days=7)
last_2week = last_week - relativedelta(days=7)
last_month = today  - relativedelta(months=1)
last_2month = last_month - relativedelta(months=1)
last_year = today - relativedelta(years=1)

first_this_year = today + relativedelta(month=1, day=1)
first_last_year = today - relativedelta(years=1, month=1, day=1)

today=today.strftime('%Y-%m-%d')
yesterday=yesterday.strftime('%Y-%m-%d')
last_week=last_week.strftime('%Y-%m-%d')
last_2week=last_2week.strftime('%Y-%m-%d')
last_month=last_month.strftime('%Y-%m-%d')
last_2month=last_2month.strftime('%Y-%m-%d')
last_year=last_year.strftime('%Y-%m-%d')
first_this_year=first_this_year.strftime('%Y-%m-%d')
first_last_year=first_last_year.strftime('%Y-%m-%d')


today = date.fromisoformat(today)
last_year = date.fromisoformat(last_year)

print('today: ', today)
print('yesterday: ', yesterday)
print('last_week: ', last_week)
print('last_2week: ', last_2week)
print('last_month: ', last_month)
print('last_2month: ', last_2month)
print('last_year: ', last_year)
print('first_this_year: ', first_this_year)
print('first_last_year: ', first_last_year)




today:  2026-04-15
yesterday:  2026-04-14
last_week:  2026-04-08
last_2week:  2026-04-01
last_month:  2026-03-15
last_2month:  2026-02-15
last_year:  2025-04-15
first_this_year:  2026-01-01
first_last_year:  2025-01-01


In [31]:
def import_sum_symptom(df,dest_table):

    if df.empty:
        print(f"DataFrame is empty. Skipping upload to {dest_table}\n")
        return  # Exit the function early
    
    # mapping pandas dtype ke BQ type
    dtype_to_bq = {
        'object': 'STRING',
        'int64': 'INT64',
        'Int64': 'INT64',
        'int32' : 'INT64',
        'float64': 'FLOAT64',
        'float32': 'FLOAT64',
        'datetime64[ns]': 'TIMESTAMP',
        'datetime64[us, UTC]':'TIMESTAMP',
        'datetime64[ns, UTC]':'TIMESTAMP',
        'dbdate':'TIMESTAMP'
    }

    # bangun schema dynamically
    schema = []
    for col, dtype in df.dtypes.items():
        dt = str(dtype)
        bq_type = dtype_to_bq.get(dt)
        if not bq_type:
            raise ValueError(f"Unknown dtype '{dt}' for column '{col}'")
        schema.append(bigquery.SchemaField(col, bq_type))

    # config load job
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition="WRITE_APPEND"        
        #write_disposition="WRITE_TRUNCATE" if i == 0 else "WRITE_APPEND"
    )

    # load ke BQ
    job = bq_client.load_table_from_dataframe(df, dest_table, job_config=job_config)
    job.result()
    print(f"loaded {job.output_rows} rows\n to", dest_table) 
    

### Tabel untuk get_customer_detail
### dm_tbl_sidik_det_suspicious_cust

#### pakai ini kalau tabel 'fds' belum dimatikan

In [8]:
#dm_det_tbl_fds_suspicious ke dm_tbl_sidik_det_suspicious_cust
sql = f"""
select *,
case
    when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
    else 'Rule-Based'
end as alert_source
from pgd-dev-data-analytics.datamart_audit.dm_det_tbl_fds_suspicious        
where detection_date = '{today}'
"""

In [9]:
df = bq_client.query(sql).to_dataframe()

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5603242 entries, 0 to 5603241
Data columns (total 15 columns):
 #   Column                   Dtype              
---  ------                   -----              
 0   symptoms                 object             
 1   cif                      object             
 2   violation_value_trx      object             
 3   violation_value_nominal  object             
 4   violation_value_gram     object             
 5   violation_value_jarak    object             
 6   violation_value_cif      object             
 7   branch_code              object             
 8   detection_date           datetime64[us, UTC]
 9   tingkat_risiko           object             
 10  skor_risiko              float64            
 11  value_trx                float64            
 12  value_nominal            float64            
 13  value_gram               float64            
 14  alert_source             object             
dtypes: datetime64[us, UTC](1), float

In [19]:
from google.cloud import bigquery
import pandas as pd

# Define your BigQuery project and table details
project_id = "pgd-dev-data-analytics"
table_id = "datamart_audit.dm_tbl_sidik_det_suspicious_cust"


# Initialize the client
# key_path = '/home/jupyterhub/SA/jupyter-pgd-dev-data-analytics-296ca984dd2f.json'
# credentials = service_account.Credentials.from_service_account_file(
#     key_path, scopes=["https://www.googleapis.com/auth/cloud-platform"],
# )
# bq_client = bigquery.Client(credentials=credentials, project=credentials.project_id)

# Configure the job to append data
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND" # Appends to an existing table
    # You can add schema definition or let BigQuery auto-detect (if table doesn't exist)
)

# Load the data
job = bq_client.load_table_from_dataframe(
    df, table_id, job_config=job_config
)

# Wait for the job to complete
job.result() 

print(f"Loaded {len(df)} rows into {table_id}.")


Loaded 5579483 rows into datamart_audit.dm_tbl_sidik_det_suspicious_cust.


### Buat tabel customer sidik

In [14]:
import sys, os
sys.path.append(os.path.join('/home/module'))
from Bq_Connect import BQDataProcessor 


In [15]:
sql = BQDataProcessor(env="dev") 

[BQDataProcessor] Connected → env=DEV, project=pgd-dev-data-analytics, SA=default (/home/jupyterhub/SA/jupyter-pgd-dev-data-analytics-296ca984dd2f.json)


In [16]:
SELECT_QUERY = """
WITH aktif_cif AS (
    SELECT DISTINCT 
        cif, 
        branch_code, 
        create_date -- This determines the type (DATETIME)
    FROM `pgd-prod-data-analytics.datalake.pgd_tbl_kredit`
    WHERE status_rek = '4'

    UNION all

    SELECT DISTINCT 
        t.cif, 
        t.branch_code, 
        CAST(NULL AS DATETIME) AS create_date -- FIX: Changed '' to NULL (Cast as DATETIME)
    FROM `pgd-prod-data-analytics.datalake.pgd_tbl_rekening_emas` e
    LEFT JOIN `pgd-prod-data-analytics.datalake.pgd_tbl_tabungan` t
        ON e.norek = t.norek
    WHERE e.saldo_akhir > 0
)
SELECT
    c.is_active,
    c.tipe_customer,
    c.flag,
    CAST(c.create_date AS TIMESTAMP)        AS create_date, 
    c.cif,
    c.nama,
    c.no_id                                 AS ktp,
    c.telp,
    c.hp,
    c.branch_code,
    CAST(c.update_date AS TIMESTAMP)        AS update_date,
    t.norek                                 AS nomor_rekening_tab_emas
FROM `pgd-prod-data-analytics.datalake.pgd_tbl_customer` c
LEFT JOIN aktif_cif a
    ON c.cif = a.cif
LEFT JOIN `pgd-prod-data-analytics.datalake.pgd_tbl_tabungan` t
    ON c.cif = t.cif
WHERE
    a.cif IS NOT NULL
    AND c.is_active = '1'
    AND c.create_date = current_date()
"""


In [17]:
DEST_TABLE = "pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_customer"    

In [18]:
sql.execute_ddl(f"""create or replace table {DEST_TABLE} as 
{SELECT_QUERY}
""")
print(sql.read(f"select * from {DEST_TABLE} limit 10"))
print(sql.read(f"select count(*) from {DEST_TABLE} limit 10"))

[DDL] Executing: create or replace table pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_customer as 

WITH aktif_cif AS (
    SELECT ...
[DDL] Success.
[READ] Executing query...
[READ] Success → 10 row(s) returned.
  is_active tipe_customer    flag                      create_date  \
0         1             1  KONVEN        2019-03-30 06:33:35+00:00   
1         1             1  KONVEN        2019-04-25 01:40:07+00:00   
2         1             1  KONVEN 2019-06-28 10:06:17.627000+00:00   
3         1             1  KONVEN        2019-07-31 02:20:26+00:00   
4         1             1  KONVEN 2019-07-10 20:37:12.949000+00:00   
5         1             1  KONVEN 2019-07-05 19:47:29.096000+00:00   
6         1             1  KONVEN 2019-02-23 18:35:05.472000+00:00   
7         1             1  KONVEN 2019-07-14 10:04:12.172000+00:00   
8         1             1  KONVEN 2019-02-28 11:24:51.324000+00:00   
9         1             1  KONVEN 2014-03-08 21:17:22.237000+00:00   

          

In [27]:
df_cus = bq_client.query(SELECT_QUERY).to_dataframe()

In [28]:
df_cus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11310475 entries, 0 to 11310474
Data columns (total 12 columns):
 #   Column                   Dtype 
---  ------                   ----- 
 0   is_active                object
 1   tipe_customer            object
 2   flag                     object
 3   create_date              object
 4   cif                      object
 5   nama                     object
 6   ktp                      object
 7   telp                     object
 8   hp                       object
 9   branch_code              object
 10  update_date              object
 11  nomor_rekening_tab_emas  object
dtypes: object(12)
memory usage: 1.0+ GB


In [29]:
df_cus['create_date'] = pd.to_datetime(df_cus['create_date'], unit='us', utc=True, errors='coerce')
df_cus['update_date'] = pd.to_datetime(df_cus['update_date'], unit='us', utc=True, errors='coerce')


In [32]:
import_sum_symptom(df_cus, DEST_TABLE)

loaded 11310475 rows
 to pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_customer


#### pakai ini kalau tabel 'fds' sudah dimatikan

##### Symptom 1a: Jumlah Rekening Kredit dalam Sehari

In [ ]:

# s_1a = f"""
# --INSERT INTO {destination_table}
# SELECT
#   'Jumlah Rekening/Kontrak Kredit dalam Sehari' AS symptoms,
#   cif,
#   CONCAT(CAST(jumlah_kontrak AS STRING), ' transaksi dalam ', periode_deteksi) AS violation_value_trx,
#   null AS violation_value_nominal,
#   null AS violation_value_gram,
#   null AS violation_value_jarak,
#   null AS violation_value_cif,
  
#   cast(jumlah_kontrak as string) as value_trx,
#   null as value_nominal,
#   null as value_gram,
  
#   cast(branch_code as string) as branch_code,
#   CAST(current_date() AS DATE) AS detection_date,
  
#   case 
#       when jumlah_kontrak <= 10 then 'Moderate'
#       when jumlah_kontrak <= 15 then 'Moderate to High'
#       when jumlah_kontrak > 15 then 'High'
#   end as tingkat_risiko,
#   case
#       when jumlah_kontrak <= 10 then 3.0
#       when jumlah_kontrak <= 15 then 4.0
#       when jumlah_kontrak > 15 then 5.0
#   end as skor_risiko,
#   case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
  
  
# FROM (
#     -- Deteksi Anomali Pencairan Kredit
#     WITH pencairan_data AS (
#         SELECT 
#             kr.cif,
#             kr.no_kontrak,
#             kr.tgl_transaksi,
#             kr.branch_code
#         FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit kr 
#         INNER JOIN pgd-prod-data-analytics.datalake.pgd_tbl_transaction_handler h 
#             ON kr.kode_transaksi = h.kode_transaksi
#         LEFT JOIN pgd-prod-data-analytics.datalake.pgd_tbl_customer cust 
#             ON kr.cif = cust.cif 
#         WHERE h.description LIKE '%CAIR%'
#             AND kr.tgl_transaksi >= date_sub(current_date(), interval 8 day)
#             AND kr.tgl_transaksi <= date_sub(current_date(), interval 1 day)
#     ),

#     anomali_1_hari AS (
#         SELECT 
#             cif,
#             branch_code,
#             COUNT(DISTINCT no_kontrak) AS jumlah_kontrak,
#             '1 Hari' AS periode_deteksi
#         FROM pencairan_data
#         WHERE tgl_transaksi = date_sub(current_date(), interval 1 day)
#         GROUP BY cif, branch_code
#         HAVING COUNT(DISTINCT no_kontrak) > 5
#     )

#     SELECT * FROM (
#         SELECT 
#             cif,
#             branch_code,
#             jumlah_kontrak,
#             periode_deteksi
#         FROM anomali_1_hari
#     ) hasil
# ) final
# """

# df = bq_client.query(s_1a).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)


##### Symptom 1b: Jumlah Rekening Kredit dalam Seminggu

In [ ]:
# s_1b = f"""
# --INSERT INTO {destination_table}
# SELECT
#   'Jumlah Rekening/Kontrak Kredit dalam Seminggu' AS symptoms,
#   cif,
#   CONCAT(CAST(jumlah_kontrak AS STRING), ' transaksi dalam ', periode_deteksi) AS violation_value_trx,
#   null AS violation_value_nominal,
#   null AS violation_value_gram,
#   null AS violation_value_jarak,
#   null AS violation_value_cif,
  
#   cast(jumlah_kontrak as string) as value_trx,
#   null as value_nominal,
#   null as value_gram,
  
#   branch_code,
#   CAST(current_date() AS DATE) AS detection_date,
#   case 
#       when jumlah_kontrak <= 20 then 'Moderate'
#       when jumlah_kontrak <= 25 then 'Moderate to High'
#       when jumlah_kontrak > 25 then 'High'
#   end as tingkat_risiko,
#   case
#       when jumlah_kontrak <= 20 then 3.0
#       when jumlah_kontrak <= 25 then 4.0
#       when jumlah_kontrak > 25 then 5.0
#   end as skor_risiko,
#   case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
  
# FROM (
#     -- Deteksi Anomali Pencairan Kredit
#     WITH pencairan_data AS (
#         SELECT 
#             kr.cif,
#             kr.no_kontrak,
#             kr.tgl_transaksi,
#             kr.branch_code
#         FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit kr 
#         INNER JOIN pgd-prod-data-analytics.datalake.pgd_tbl_transaction_handler h 
#             ON kr.kode_transaksi = h.kode_transaksi
#         LEFT JOIN pgd-prod-data-analytics.datalake.pgd_tbl_customer cust 
#             ON kr.cif = cust.cif 
#         WHERE h.description LIKE '%CAIR%'
#             AND kr.tgl_transaksi >= date_sub(current_date(), interval 8 day)
#             AND kr.tgl_transaksi <= date_sub(current_date(), interval 1 day)
#     ),

#     anomali_1_hari AS (
#         SELECT 
#             cif,
#             branch_code,
#             COUNT(DISTINCT no_kontrak) AS jumlah_kontrak,
#             '1 Hari' AS periode_deteksi
#         FROM pencairan_data
#         WHERE tgl_transaksi = date_sub(current_date(), interval 1 day)
#         GROUP BY cif, branch_code
#         HAVING COUNT(DISTINCT no_kontrak) > 5
#     ),

#     anomali_7_hari AS (
#         SELECT 
#             pd.cif,
#             pd.branch_code,
#             COUNT(DISTINCT pd.no_kontrak) AS jumlah_kontrak,
#             '7 Hari' AS periode_deteksi
#         FROM pencairan_data pd
#         LEFT JOIN anomali_1_hari a1
#             ON pd.cif = a1.cif
#         WHERE a1.cif IS NULL
#             AND pd.tgl_transaksi BETWEEN date_sub(current_date(), interval 7 day) AND date_sub(current_date(), interval 1 day)
#         GROUP BY pd.cif, pd.branch_code
#         HAVING COUNT(DISTINCT pd.no_kontrak) > 15
#     )

#     SELECT * FROM (
#         SELECT 
#             cif,
#             branch_code,
#             jumlah_kontrak,
#             periode_deteksi
#         FROM anomali_7_hari
#     ) hasil
# ) final
# """

# df = bq_client.query(s_1b).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)


##### Symptom 2: Jumlah Outlet Transaksi Kredit di Luar Kewajaran

In [ ]:
# s_2 = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Outlet Transaksi Kredit di Luar Kewajaran' as symptoms,
# base.cif,
# concat('transaksi di ', cast(base.jumlah_outlet as STRING), ' outlet dalam 1 hari') as violation_value_trx,
# null as violation_value_nominal,
# null as violation_value_gram,
# null AS violation_value_jarak,
# null AS violation_value_cif,

# cast (base.jumlah_outlet as string) as value_trx,
# null as value_nominal,
# null as value_gram,

# a.branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# case 
#   when base.jumlah_outlet <= 3 then 'Moderate' 
#   when base.jumlah_outlet <= 4 then 'Moderate to High'
#   when base.jumlah_outlet > 4 then 'High'
# end as tingkat_risiko,
# case
#   when base.jumlah_outlet <= 3 then 3.0
#   when base.jumlah_outlet <= 4 then 4.0
#   when base.jumlah_outlet > 4 then 5.0
# end as skor_risiko,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source


# from(    
#     SELECT
#       a.cif,
#       a.tgl_transaksi,
#       COUNT(DISTINCT a.branch_code) AS jumlah_outlet
#     FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit a
#     JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch b
#         ON a.branch_code = b.branch_cd
#     join pgd-prod-data-analytics.datalake.pgd_tbl_transaction_handler h
#         on a.kode_transaksi = h.kode_transaksi
#     where
#         description like '%CAIR%' -- ini buat ngambil yang terkait dengan 'pencairan'
#         and (description like '%GADAI%' or description like '%RAHN%' or description like '%MIKRO%') -- ini buat ngambil gadai dan mikro exclude tab emas/gte
#         and tgl_transaksi = date_sub(current_date(), interval 1 day)
#     GROUP BY a.cif, a.tgl_transaksi
#     HAVING COUNT(DISTINCT a.branch_code) > 2 -- ganti limit
#     ORDER BY jumlah_outlet DESC) base
# inner join pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit a
#     on base.cif = a.cif
#     and base.tgl_transaksi = a.tgl_transaksi
# """
# df = bq_client.query(s_2).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

##### Symptom 3: Jumlah Nominal Kredit & Ratio OSL Nasabah dibandingkan OutletJumlah Nominal Kredit & Ratio OSL Nasabah dibandingkan Outlet

###### Nilai kredit aktif nasabah yang melebihi Rp.120juta dan memiliki total OSL aktif yang dimiliki nasabah > 5% dari total OSL di Collocation

In [ ]:
# s_3a = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Nominal Kredit & Ratio OSL Nasabah dibandingkan Outlet dalam Rentang Tertentu' as symptoms,
# base.cif,
# null as violation_value_trx,
# concat('Jumlah Kredit Nasabah di Unit Kerja Collocation ', cast(base.total_nilai_kredit as STRING), ' dengan OSL aktif >5% di Collocation') as violation_value_nominal,
# null as violation_value_gram,
# null AS violation_value_jarak,
# null AS violation_value_cif,

# null as value_trx,
# cast (base.total_nilai_kredit as string) as value_nominal,
# null as value_gram,

# base.branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# case 
#   when base.total_nilai_kredit <= 350000000 then 'Moderate'
#   when base.total_nilai_kredit <= 530000000 then 'Moderate to High'
#   when base.total_nilai_kredit > 530000000 then 'High'
# end as tingkat_risiko,
# case
#   when base.total_nilai_kredit <= 350000000 then 3.0
#   when base.total_nilai_kredit <= 530000000 then 4.0
#   when base.total_nilai_kredit > 530000000 then 5.0
# end as skor_risiko,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source


# from(    
#     WITH total_osl_collocation AS (
#       SELECT
#         a.branch_code,
#         SUM(b.up) AS total_osl_collocation
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit a
#       JOIN pgd-prod-data-analytics.datalake.pgd_tbl_kredit b
#         ON a.cif = b.cif
#       JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch c
#         ON CAST(a.branch_code AS STRING) = c.sub_branch_cd 
#       WHERE b.status_rek = '4' 
#           AND (a.product_code = '01' OR a.product_code = '02')
#           AND c.sub_branch_nm LIKE '%BRI%'
#       GROUP BY a.branch_code
#     ),
#     nasabah_summary AS (
#       SELECT
#         a.cif,
#         a.branch_code,
#         SUM(a.up) AS total_nilai_kredit
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_kredit a
#       JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch b
#         ON CAST(a.branch_code AS STRING) = b.sub_branch_cd
#       WHERE 
#         a.status_rek = '4' 
#         AND (a.product_code = '01' OR a.product_code = '02')
#         AND b.sub_branch_nm LIKE '%BRI%' 
#       GROUP BY a.cif, a.branch_code
#     )

#     SELECT DISTINCT
#       n.cif,
#       n.branch_code,
#       b.region_nm,  
#       n.total_nilai_kredit,
#       t.total_osl_collocation,
#       ROUND(100 * n.total_nilai_kredit / t.total_osl_collocation, 2) AS persen_kontribusi_osl
#     FROM nasabah_summary n
#     JOIN total_osl_collocation t
#       ON n.branch_code = t.branch_code
#     JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch b  
#       ON n.branch_code = b.sub_branch_cd
#     WHERE n.total_nilai_kredit > 120000000
#       AND (n.total_nilai_kredit / t.total_osl_collocation) > 0.05
#     ORDER BY persen_kontribusi_osl DESC)
#     AS base
# """
# df = bq_client.query(s_3a).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

###### nilai kredit aktif nasabah yang ***melebihi Rp.300juta*** dan memiliki total OSL aktif yang dimiliki nasabah > 5% dari total OSL di UPC/UPS

In [ ]:
# s_3b = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Nominal Kredit & Ratio OSL Nasabah dibandingkan Outlet dalam Rentang Tertentu' as symptoms,
# base.cif,
# null as violation_value_trx,
# concat('Jumlah Kredit Nasabah di UPC/UPS ', cast(base.total_nilai_kredit as STRING), ' dengan OSL aktif >5% di UPC/UPS') as violation_value_nominal,
# null as violation_value_gram,
# null AS violation_value_jarak,
# null AS violation_value_cif,
# base.branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# null as value_trx,
# cast (base.total_nilai_kredit as string) as value_nominal,
# null as value_gram,


# case 
#   when base.total_nilai_kredit <= 400000000 then 'Moderate'
#   when base.total_nilai_kredit <= 500000000 then 'Moderate to High'
#   when base.total_nilai_kredit > 500000000 then 'High'
# end as tingkat_risiko,
# case
#   when base.total_nilai_kredit <= 400000000 then 3.0
#   when base.total_nilai_kredit <= 500000000 then 4.0
#   when base.total_nilai_kredit > 500000000 then 5.0
# end as skor_risiko,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source


# from(    
#     WITH total_osl_upc AS (
#       SELECT
#         a.branch_code,
#         SUM(b.up) AS total_osl_upc
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit a
#       JOIN pgd-prod-data-analytics.datalake.pgd_tbl_kredit b
#         ON a.cif = b.cif
#       JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch c
#         ON CAST(a.branch_code AS STRING) = c.sub_branch_cd 
#       WHERE b.status_rek = '4' 
#           AND (a.product_code = '01' OR a.product_code = '02')
#           AND c.sub_branch_nm NOT LIKE '%BRI%'
#           AND c.hirarki = 'UPC'
#       GROUP BY a.branch_code
#     ),
#     nasabah_summary AS (
#       SELECT
#         a.cif,
#         a.branch_code,
#         SUM(a.up) AS total_nilai_kredit
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_kredit a
#       JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch b
#         ON CAST(a.branch_code AS STRING) = b.sub_branch_cd
#       WHERE 
#         a.status_rek = '4' 
#         AND (a.product_code = '01' OR a.product_code = '02')
#         AND b.sub_branch_nm NOT LIKE '%BRI%'
#         AND b.hirarki ='UPC'
#       GROUP BY a.cif, a.branch_code
#     )

#     SELECT DISTINCT
#       n.cif,
#       n.branch_code,
#       b.region_nm,  
#       n.total_nilai_kredit,
#       t.total_osl_upc,
#       ROUND(100 * n.total_nilai_kredit / t.total_osl_upc, 2) AS persen_kontribusi_osl
#     FROM nasabah_summary n
#     JOIN total_osl_upc t
#       ON n.branch_code = t.branch_code
#     JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch b  
#       ON n.branch_code = b.sub_branch_cd
#     WHERE n.total_nilai_kredit > 300000000 -- limit 300jt
#       AND (n.total_nilai_kredit / t.total_osl_upc) > 0.05 --- limit 5%
#     ORDER BY persen_kontribusi_osl DESC)
#     AS base
# """
# df = bq_client.query(s_3b).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

###### Nilai kredit aktif nasabah yang melebihi Rp.1 Miliar di CP

In [ ]:
# s_3c = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Nominal Kredit & Ratio OSL Nasabah dibandingkan Outlet dalam Rentang Tertentu' as symptoms,
# base.cif,
# null as violation_value_trx,
# concat('Jumlah Kredit Nasabah di CP/CPS ', cast(base.total_nilai_kredit as STRING), ' diatas 1M') as violation_value_nominal,
# null as violation_value_gram,
# null AS violation_value_jarak,
# null AS violation_value_cif,

# null as value_trx,
# cast (base.total_nilai_kredit as string) as value_nominal,
# null as value_gram,

# base.branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# case 
#   when base.total_nilai_kredit <= 1500000000 then 'Moderate'
#   when base.total_nilai_kredit <= 2000000000 then 'Moderate to High'
#   when base.total_nilai_kredit > 2000000000 then 'High'
# end as tingkat_risiko,
# case
#   when base.total_nilai_kredit <= 1500000000 then 3.0
#   when base.total_nilai_kredit <= 2000000000 then 4.0
#   when base.total_nilai_kredit > 2000000000 then 5.0
# end as skor_risiko,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source


# from(    
#     WITH nasabah_summary AS (
#   SELECT
#     a.cif,
#     a.branch_code,
#     b.hirarki,
#     SUM(a.up) AS total_nilai_kredit
#   FROM pgd-prod-data-analytics.datalake.pgd_tbl_kredit a
#   JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch b
#     ON a.branch_code = b.sub_branch_cd
#   WHERE 
#     a.status_rek = '4' 
#     AND (a.product_code = '01' OR a.product_code = '02')
#     AND b.hirarki ='CABANG'
#   GROUP BY a.cif, a.branch_code, b.hirarki
# )

# SELECT DISTINCT
#   n.cif,
#   n.branch_code,
#   b.region_nm,  
#   n.total_nilai_kredit,
#   n.hirarki
# FROM nasabah_summary n
# JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch b  
#   ON n.branch_code = b.sub_branch_cd
# WHERE n.total_nilai_kredit > 1000000000)
#     AS base
# """
# df = bq_client.query(s_3c).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

##### Symptom 4: Jumlah Nominal Kredit dalam Sehari & Seminggu

In [ ]:
# s_4a = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Nominal Kredit dalam Sehari' as symptoms,
# base.cif,
# null as violation_value_trx,
# concat('Jumlah nominal kredit ', cast(base.total_up_pada_hari_tersebut as STRING), ' dalam 1 hari 5 kontrak') as violation_value_nominal,
# null as violation_value_gram,
# null AS violation_value_jarak,
# null AS violation_value_cif,

# null as value_trx,
# cast (base.total_up_pada_hari_tersebut as string) as value_nominal,
# null as value_gram,

# base.branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# case 
#   when base.total_up_pada_hari_tersebut <= 100000000 then 'Moderate'
#   when base.total_up_pada_hari_tersebut <= 300000000 then 'Moderate to High'
#   when base.total_up_pada_hari_tersebut > 300000000 then 'High'
# end as tingkat_risiko,
# case
#   when base.total_up_pada_hari_tersebut <= 100000000 then 3.0
#   when base.total_up_pada_hari_tersebut <= 300000000 then 4.0
#   when base.total_up_pada_hari_tersebut > 300000000 then 5.0
# end as skor_risiko,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source


# from(    
#     WITH kredit_base AS (
#       -- Ambil data transaksi kredit aktif dan nominal UP
#       SELECT
#         t.cif,
#         t.no_kontrak,
#         k.up,
#         k.branch_code,
#         CAST(t.tgl_transaksi AS DATE) AS transaction_date
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit t
#       JOIN pgd-prod-data-analytics.datalake.pgd_tbl_kredit k
#         ON t.no_kontrak = k.no_kontrak
#       WHERE k.status_rek = '4'
#         AND k.up > 2000000 -- Filter UP > 2 Juta di awal untuk efisiensi
#     )
#     SELECT
#       cif,
#       transaction_date,
#       COUNT(no_kontrak) AS jumlah_kontrak_up_2jt,
#       MIN(up) AS min_up_pada_hari_tersebut,
#       MAX(up) AS max_up_pada_hari_tersebut,
#       SUM(up) AS total_up_pada_hari_tersebut,
#       branch_code
#     FROM kredit_base
#     WHERE transaction_date >= DATE_SUB(CURRENT_DATE(), interval 1 day) -- Hanya sehari
#     GROUP BY cif, transaction_date, branch_code
#     HAVING COUNT(no_kontrak) > 5 -- Hanya yang memiliki lebih dari 5 kontrak
#     --ORDER BY transaction_date DESC, jumlah_kontrak_up_2jt DESC
#     )
#     AS base
# """
# df = bq_client.query(s_4a).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

In [ ]:
# s_4b = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Nominal Kredit dalam Seminggu' as symptoms,
# base.cif,
# null as violation_value_trx,
# concat('Jumlah nominal kredit ', cast(base.total_up_7hari as STRING), ' dalam 7 hari 15 kontrak') as violation_value_nominal,
# null as violation_value_gram,
# null AS violation_value_jarak,
# null AS violation_value_cif,

# null as value_trx,
# cast (base.total_up_7hari as string) as value_nominal,
# null as value_gram,

# base.branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# case 
#   when base.total_up_7hari <= 150000000 then 'Moderate'
#   when base.total_up_7hari <= 300000000 then 'Moderate to High'
#   when base.total_up_7hari > 300000000 then 'High'
# end as tingkat_risiko,
# case
#   when base.total_up_7hari <= 150000000 then 3.0
#   when base.total_up_7hari <= 300000000 then 4.0
#   when base.total_up_7hari > 300000000 then 5.0
# end as skor_risiko,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source


# from(    
#     WITH kredit_base AS (
#       -- Ambil data transaksi kredit aktif dan nominal UP
#       SELECT
#         t.cif,
#         t.no_kontrak,
#         k.up,
#         k.branch_code,
#         CAST(t.tgl_transaksi AS DATE) AS transaction_date
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit t
#       JOIN pgd-prod-data-analytics.datalake.pgd_tbl_kredit k
#         ON t.no_kontrak = k.no_kontrak
#       WHERE k.status_rek = '4'
#         AND k.up > 7500000 -- Filter UP > 2 Juta di awal 
#     )
#     SELECT
#       cif,
#       COUNT(no_kontrak) AS jumlah_kontrak_up_7_5jt_7hari,
#       MIN(up) AS min_up_7hari,
#       MAX(up) AS max_up_7hari,
#       SUM(up) AS total_up_7hari,
#       branch_code
#     FROM kredit_base t
#     WHERE t.transaction_date >= DATE_SUB(CURRENT_DATE(), interval 7 day) -- Hanya 7 hari kalender terakhir
#     GROUP BY cif, branch_code
#     HAVING COUNT(no_kontrak) > 15 -- Hanya yang memiliki lebih dari 15 kontrak
#     ORDER BY jumlah_kontrak_up_7_5jt_7hari DESC)
#     AS base
# """
# df = bq_client.query(s_4b).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

##### Symptom 6: Jumlah CIF Nasabah dengan Nomor KTP/Passport yang Sama di Luar Kewajaran

In [ ]:
# s_6 = f"""
# --INSERT INTO {destination_table}
# WITH aktif_cif AS (
#       SELECT DISTINCT cif, branch_code, safe_cast(create_date as timestamp) as create_date
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_kredit
#       WHERE status_rek = '4' 
      
#       union all
      
#       SELECT DISTINCT t.cif, t.branch_code, safe_cast('' as timestamp) as create_date
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_rekening_emas e
#       left join pgd-prod-data-analytics.datalake.pgd_tbl_tabungan t
#       ON e.norek = t.norek
#       where 
#       e.saldo_akhir > 0
      
#     ),
#     ktp_cleaned AS (
#       SELECT DISTINCT
#         c.cif,
#         c.nama,
#         a.branch_code,
#         a.create_date,
#         REGEXP_REPLACE(c.no_id, '[^0-9]', '') AS clean_ktp
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_customer c
#       JOIN aktif_cif a
#         ON c.cif = a.cif
#       WHERE c.no_id IS NOT NULL 
#           and trim(c.no_id) != ''
#           and length(c.no_id) = 16
#           AND c.no_id NOT LIKE '%000000%'
#           AND c.no_id NOT LIKE '%1111111%'
#           AND c.no_id NOT LIKE '%X%'
#           AND c.is_active = '1' --- 
#     ),
#     ktp_group AS (
#       SELECT
#         clean_ktp,
#         COUNT(DISTINCT cif) AS jumlah_cif
#       FROM ktp_cleaned
#       GROUP BY clean_ktp
#       HAVING COUNT(DISTINCT cif) > 2
#     )
#     SELECT
#       'Jumlah CIF Nasabah dengan Nomor KTP/Passport yang Sama di Luar Kewajaran' as symptoms,
#       k.cif,
#       concat(cast(jumlah_cif as string), ' CIF dalam 1 KTP')  as violation_value_trx,
#       null AS violation_value_nominal,
#       null AS violation_value_gram,
#       null AS violation_value_jarak,
#       null AS violation_value_cif,
      
#       null as value_trx,
#       null as value_nominal,
#       null as value_gram,
      
#       k.branch_code,
#       CAST(current_date() AS DATE) AS detection_date,
      
#       case 
#           when jumlah_cif <= 3 then 'Moderate'
#           when jumlah_cif <= 4 then 'Moderate to High'
#           when jumlah_cif > 4 then 'High'
#       end as tingkat_risiko,
#       case
#           when jumlah_cif <= 3 then 3.0
#           when jumlah_cif <= 4 then 4.0
#           when jumlah_cif > 4 then 5.0
#       end as skor_risiko,
#       case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
      
      
#     FROM ktp_group g
#     JOIN ktp_cleaned k
#       ON g.clean_ktp = k.clean_ktp
#     left join pgd-prod-data-analytics.datalake.pgd_tbl_customer c
#       on k.cif = c.cif
#     left join pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch vt
#         on k.branch_code = vt.sub_branch_cd
# """
# df = bq_client.query(s_6).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

##### Symptom 7: Jumlah Frekuensi Buyback dalam 10 menit


In [ ]:
# s_7a = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Frekuensi TopUp dan Buyback dalam 10 menit' as symptoms,
# base.cif,
# concat(cast(base.jumlah_buyback_cepat as STRING), ' kali transaksi Buyback TE per nasabah dalam kurun 10 menit', '') as violation_value_trx,
# null as violation_value_nominal,
# null as violation_value_gram,
# null as violation_value_jarak,
# null as violation_value_cif, --buat symptom ktp, device_id, no hp

# cast (base.jumlah_buyback_cepat as string) as value_trx,
# null as value_nominal,
# null as value_gram,

# base.branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# case 
#   when base.jumlah_buyback_cepat <= 2 then 'Moderate'
#   when base.jumlah_buyback_cepat = 3 then 'Moderate to High'
#   when base.jumlah_buyback_cepat > 3 then 'High'
# end as tingkat_risiko,
# case
#   when base.jumlah_buyback_cepat <= 2 then 3.0
#   when base.jumlah_buyback_cepat = 3 then 4.0
#   when base.jumlah_buyback_cepat > 3 then 5.0
# end as skor_risiko,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
# from(    
#     WITH trx_ranked AS (
#       SELECT
#         t.id,
#         t.cif,
#         t.jenis_transaksi,
#         t.create_date,
#         d.berat,
#         t.branch_code,
#         ROW_NUMBER() OVER (PARTITION BY t.cif ORDER BY t.create_date) as rn,
#         LEAD(t.jenis_transaksi, 1) OVER (PARTITION BY t.cif ORDER BY t.create_date) AS next_jenis,
#         LEAD(t.create_date, 1) OVER (PARTITION BY t.cif ORDER BY t.create_date) AS next_tgl,
#         LEAD(d.berat, 1) OVER (PARTITION BY t.cif ORDER BY t.create_date) AS next_berat
#       FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_tabungan t
#       JOIN pgd-prod-data-analytics.datalake.pgd_tbl_e_transaksi_emas_det d
#         ON t.id = d.id
#       WHERE t.jenis_transaksi IN ('BUY', 'SALE') 
#     )
#     SELECT
#       cif,
#       COUNT(CASE WHEN TIMESTAMP_DIFF(next_tgl, create_date, SECOND) <= 600 THEN 1 END) AS jumlah_buyback_cepat,
#       MIN(ROUND(TIMESTAMP_DIFF(next_tgl, create_date, SECOND) / 60.0, 2)) AS waktu_tercepat_menit,
#       MAX(CASE WHEN TIMESTAMP_DIFF(next_tgl, create_date, SECOND) <= 600 THEN berat END) AS max_berat_buy,
#       MAX(CASE WHEN TIMESTAMP_DIFF(next_tgl, create_date, SECOND) <= 600 THEN next_berat END) AS max_berat_sale,
#       create_date,
#       branch_code
#     FROM trx_ranked
#     WHERE
#         jenis_transaksi = 'SALE'
#         AND next_jenis = 'BUY'
#     GROUP BY cif, create_date, branch_code
#     having jumlah_buyback_cepat > 0)
#     AS base
    
# where create_date = date_sub(current_date(), interval 1 day) -- ambil hari yang kemarin aja; biar gak terlalu banyak
# """
# df = bq_client.query(s_7a).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

##### Symptom 8a: Jumlah Frekeunsi Buyback dalam Sehari

In [ ]:
# s_7b = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Frekuensi TopUp dan Buyback dalam Sehari' as symptoms,
# base.cif,
# concat(cast(base.total_buybacks_under_1_day as STRING), ' kali transaksi Buyback TE per nasabah dalam kurun sehari', '') as violation_value_trx,
# null as violation_value_nominal,
# null as violation_value_gram,
# null as violation_value_jarak,
# null as violation_value_cif, --buat symptom ktp, device_id, no hp

# cast(base.total_buybacks_under_1_day as string) as value_trx,
# null as value_nominal,
# null as value_gram,

# base.branch_code,
# current_date() AS detection_date,
# case 
#   when base.total_buybacks_under_1_day <= 50 then 'Moderate'
#   when base.total_buybacks_under_1_day <= 100 then 'Moderate to High'
#   when base.total_buybacks_under_1_day > 100 then 'High'
# end as tingkat_risiko,
# case
#   when base.total_buybacks_under_1_day <= 50 then 3.0
#   when base.total_buybacks_under_1_day <= 100 then 4.0
#   when base.total_buybacks_under_1_day > 100 then 5.0
# end as skor_risiko,
# base.latest_buyback_date,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
# from(    
#     WITH trx_ranked AS (
#     -- 1. Identifikasi pasangan BUY diikuti SALE menggunakan Window Function
#     SELECT
#         t.cif,
#         t.create_date,
#         d.berat AS berat_buy,
#         t.jenis_transaksi,
#         t.branch_code,
#         LEAD(t.jenis_transaksi, 1) OVER (PARTITION BY t.cif ORDER BY t.create_date) AS next_jenis,
#         LEAD(t.create_date, 1) OVER (PARTITION BY t.cif ORDER BY t.create_date) AS sale_time,
#         LEAD(d.berat, 1) OVER (PARTITION BY t.cif ORDER BY t.create_date) AS berat_sale
#     FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_tabungan t
#     JOIN pgd-prod-data-analytics.datalake.pgd_tbl_e_transaksi_emas_det d
#         ON t.id = d.id
#     WHERE t.jenis_transaksi IN ('BUY', 'SALE') 
#         and date(create_date) = date_sub(current_date(), interval 1 day)
#     ),
#     buyback_pairs AS (
#         -- 2. Filter hanya untuk pasangan SALE diikuti BUY dengan gap < 1 hari
#         SELECT
#             cif,
#             sale_time,
#             berat_buy,
#             berat_sale,
#             branch_code,
#             -- Selisih waktu dalam detik
#             timestamp_diff(sale_time, create_date, second) AS diff_seconds
#         FROM trx_ranked
#         WHERE
#             jenis_transaksi = 'SALE'
#             AND next_jenis = 'BUY'
#             AND timestamp_diff(sale_time, create_date, second) < 86400 --GAP KURANG DARI 1 HARI

#     )
#     -- 3. Hitung dan filter hasil akhir
#     SELECT
#         cif,
#         branch_code,
#         COUNT(*) AS total_buybacks_under_1_day,
#         MAX(berat_buy) AS max_berat_buy,
#         MAX(berat_sale) AS max_berat_sale,
#         MIN(ROUND(diff_seconds / 60, 2)) AS fastest_gap_minutes,
#         MAX(CAST(sale_time AS DATE)) AS latest_buyback_date
#     FROM buyback_pairs
#     GROUP BY cif, branch_code
#     HAVING COUNT(*) > 5 -- KRITERIA HITUNGAN > 5
#     )
#     AS base
# """
# df = bq_client.query(s_7b).to_dataframe()

# df['latest_buyback_date']=pd.to_datetime(df['latest_buyback_date'])
# df=df[df['latest_buyback_date']>(pd.Timestamp.now()-pd.Timedelta(days=2))]
# df=df.drop('latest_buyback_date', axis=1)
# df=df[df['cif']!='']
# df=df.drop_duplicates()
# df['detection_date'] = pd.to_datetime(df['detection_date'], errors='coerce')
# df['skor_risiko'] = df['skor_risiko'].astype(float)

# print('total rows: ', len(df))
# print(df.head(2))


# import_detail_symptom(df,destination_table)

##### Symptom 8b: Jumlah Frekeunsi Buyback dalam Seminggu

In [ ]:
# s_7c = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Frekuensi TopUp dan Buyback dalam Seminggu' as symptoms,
# base.cif,
# concat(cast(base.total_buybacks_under_7_day as STRING), ' kali transaksi Buyback TE per nasabah dalam kurun seminggu', '') as violation_value_trx,
# null as violation_value_nominal,
# null as violation_value_gram,
# null as violation_value_jarak,
# null as violation_value_cif, --buat symptom ktp, device_id, no hp

# cast (base.total_buybacks_under_7_day as string) as value_trx,
# null as value_nominal,
# null as value_gram,

# base.branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# case 
#   when base.total_buybacks_under_7_day <= 80 then 'Moderate'
#   when base.total_buybacks_under_7_day = 140 then 'Moderate to High'
#   when base.total_buybacks_under_7_day > 140 then 'High'
# end as tingkat_risiko,
# case
#   when base.total_buybacks_under_7_day <= 80 then 3.0
#   when base.total_buybacks_under_7_day = 140 then 4.0
#   when base.total_buybacks_under_7_day > 140 then 5.0
# end as skor_risiko,
# base.latest_buyback_date,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
# from(    
#     WITH trx_ranked AS (
#     -- 1. Identifikasi pasangan BUY diikuti SALE menggunakan Window Function
#     SELECT
#         t.cif,
#         t.create_date,
#         d.berat AS berat_buy,
#         t.jenis_transaksi,
#         t.branch_code,
#         LEAD(t.jenis_transaksi, 1) OVER (PARTITION BY t.cif ORDER BY t.create_date) AS next_jenis,
#         LEAD(t.create_date, 1) OVER (PARTITION BY t.cif ORDER BY t.create_date) AS sale_time,
#         LEAD(d.berat, 1) OVER (PARTITION BY t.cif ORDER BY t.create_date) AS berat_sale
#     FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_tabungan t
#     JOIN pgd-prod-data-analytics.datalake.pgd_tbl_e_transaksi_emas_det d
#         ON t.id = d.id
#     WHERE t.jenis_transaksi IN ('BUY', 'SALE') and date(create_date) >= date_sub(current_date(), interval 7 day)
# ),
# buyback_pairs AS (
#     -- 2. Filter hanya untuk pasangan BUY diikuti SALE dengan gap < 1 hari
#     SELECT
#         cif,
#         sale_time,
#         berat_buy,
#         berat_sale,
#         branch_code,
#         -- Selisih waktu dalam detik
#         timestamp_diff(sale_time, create_date, second) AS diff_seconds
#     FROM trx_ranked
#     WHERE
#         jenis_transaksi = 'SALE'
#         AND next_jenis = 'BUY'
#         AND timestamp_diff(sale_time, create_date, second) < 604800 -- GAP KURANG DARI 7 HARI
# )
# -- 3. Hitung dan filter hasil akhir
# SELECT
#     cif,
#     branch_code,
#     COUNT(*) AS total_buybacks_under_7_day,
#     MAX(berat_buy) AS max_berat_buy,
#     MAX(berat_sale) AS max_berat_sale,
#     MIN(ROUND(diff_seconds / 60, 2)) AS fastest_gap_minutes,
#     MAX(CAST(sale_time AS DATE)) AS latest_buyback_date
# FROM buyback_pairs
# GROUP BY cif, branch_code
# HAVING COUNT(*) > 20 -- KRITERIA HITUNGAN > 20
#     )
#     AS base
# """
# df_s_7c = bq_client.query(s_7c).to_dataframe()
# df_s_7c['latest_buyback_date']=pd.to_datetime(df_s_7c['latest_buyback_date'])
# df_s_7c=df_s_7c[df_s_7c['latest_buyback_date']>(pd.Timestamp.now()-pd.Timedelta(days=8))]
# df_s_7c=df_s_7c.drop('latest_buyback_date', axis=1)
# df_s_7c=df_s_7c[df_s_7c['cif']!='']
# df_s_7c=df_s_7c.drop_duplicates()
# df_s_7c['detection_date'] = pd.to_datetime(df_s_7c['detection_date'], errors='coerce')
# df_s_7c['skor_risiko'] = df_s_7c['skor_risiko'].astype(float)
# df=df_s_7c

# print('total rows: ', len(df))
# print(df.head(2))

# import_detail_symptom(df,destination_table)

##### Symptom 9: Gramasi Buyback Mendekati Batas Maksimum

In [ ]:
# s_7d = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Gramasi Buyback Mendekati Batas Maksimum' as symptoms,
# base.cif,
# concat('Jumlah transaksi Buyback TE per nasabah dalam kurun 1 hari sebanyak ', cast(base.total_buyback_cycles_1_day as STRING),'') as violation_value_trx,
# null as violation_value_nominal,
# concat( '1 kali transaksi dengan gramasi maksimum ', cast(base.max_berat_sale_single_trx as STRING)) as violation_value_gram,
# null as violation_value_jarak,
# null as violation_value_cif, --buat symptom ktp, device_id, no hp

# cast (base.total_buyback_cycles_1_day as string) as value_trx,
# null as value_nominal,
# cast (base.max_berat_sale_single_trx as string) as value_gram,

# base.branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# base.tingkat_risiko,
# base.skor_risiko
# from(    
#     WITH trx_details AS (
#     SELECT
#         t.cif,
#         t.jenis_transaksi,
#         t.create_date,
#         CAST(t.create_date AS DATE) AS transaction_date,
#         t.branch_code,
#         det.berat
#     FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_tabungan t
#     JOIN pgd-prod-data-analytics.datalake.pgd_tbl_e_transaksi_emas_det det
#         ON t.id = det.id
#     WHERE t.jenis_transaksi IN ('BUY', 'SALE') and date(create_date) >= date_sub(current_date(), interval 7 day)
# ),
# buyback_pairs AS (
#     SELECT
#         cif,
#         create_date AS buy_time,
#         berat AS berat_buy,
#         branch_code,
#         jenis_transaksi,
#         -- Window functions untuk mendapatkan data SALE berikutnya
#         LEAD(jenis_transaksi, 1) OVER (PARTITION BY cif ORDER BY create_date) AS next_jenis,
#         LEAD(create_date, 1) OVER (PARTITION BY cif ORDER BY create_date) AS sale_time,
#         LEAD(berat, 1) OVER (PARTITION BY cif ORDER BY create_date) AS berat_sale
#     FROM trx_details
# ),
# validated_buybacks AS (
#     SELECT
#         cif,
#         buy_time,
#         sale_time,
#         CAST(sale_time AS DATE) AS buyback_date,
#         berat_buy,
#         berat_sale,
#         jenis_transaksi,
#         -- Jarak waktu (dalam jam)
#         ROUND((timestamp_diff(sale_time, buy_time, second)) / 3600, 2) AS diff_hours,
#         -- Perbedaan Gramasi Relatif: (ABS(Buy - Sale) / Buy)
#         safe_divide(ABS(berat_buy - berat_sale), berat_buy) AS berat_diff_ratio,
#         branch_code
#     FROM buyback_pairs
#     WHERE
#         jenis_transaksi = 'SALE'
#         AND next_jenis = 'BUY'
#         -- Batasan waktu berdekatan (1 jam)
#         AND timestamp_diff(sale_time, buy_time, second) <= 3600
#         -- Gramasi relatif sama
#         AND safe_divide(ABS(berat_buy - berat_sale), berat_buy) <= 0.05
#         and safe_divide(ABS(berat_buy - berat_sale), berat_buy) is not null
# ),
# accumulated_anomaly AS (
#     SELECT
#         v.cif,
#         v.buyback_date,
#         SUM(v.berat_sale) AS total_gramasi_buyback_harian,
#         -- Window function untuk akumulasi 3 hari terakhir (jika dibutuhkan)
#         SUM(SUM(v.berat_sale)) OVER (
#             PARTITION BY v.cif 
#             ORDER BY v.buyback_date 
#             ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
#         ) AS total_gramasi_buyback_3_hari,
#         MAX(v.berat_sale) AS max_berat_sale_single_trx,
#         COUNT(*) AS total_buyback_cycles_1_day,
#         MAX(v.branch_code) AS representative_branch_code
#     FROM validated_buybacks v
#     GROUP BY v.cif, v.buyback_date
# )
# SELECT
#     a.cif,
#     a.buyback_date,
#     b.region_nm,
#     a.total_buyback_cycles_1_day,
#     a.total_gramasi_buyback_harian,
#     a.total_gramasi_buyback_3_hari,
#     a.max_berat_sale_single_trx,
#     a.representative_branch_code AS branch_code,
#     -- Flag Deteksi: Total gramasi mendekati batas maksimum
#     CASE 
#         WHEN a.total_gramasi_buyback_harian >= 0.95 * 100 THEN 'High'
#         WHEN a.total_gramasi_buyback_3_hari >= 0.95 * 100 THEN 'Moderate to High'
#         ELSE 'Moderate'
#     END AS tingkat_risiko,
#     CASE 
#         WHEN a.total_gramasi_buyback_harian >= 0.95 * 100 THEN 3.0
#         WHEN a.total_gramasi_buyback_3_hari >= 0.95 * 100 THEN 4.0
#         ELSE 5.0
#     END AS skor_risiko,
#     case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
# FROM accumulated_anomaly a
# LEFT JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch b
#     ON a.representative_branch_code = b.branch_cd
# WHERE
#     -- Filter hanya yang total buyback hariannya atau 3 harinya mendekati batas maksimum
#     a.total_gramasi_buyback_harian >= 0.50 * 100 -- Ambil di atas 50% dari batas
#     OR a.total_gramasi_buyback_3_hari >= 0.70 * 100 -- Ambil di atas 70% dari batas 3 hari
# )
#     AS base
# """
# df = bq_client.query(s_7d).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

##### Symptom 10: Rentang Jarak Pencairan Gadai Melalui Outlet diluar kewajaran (sek kok ora cetho)

In [ ]:
# import pandas as pd
# import math

# #colo =[sub_branch_cd, sub_branch_nm, lat, long]


# count_query = f"""
#     with ct_row as (
#            SELECT distinct 
#   cif,
#   no_kontrak,
#   up,
#   k.create_date,
#   branch_code,
#   g.latitude,
#   g.longitude
# FROM
#   pgd-prod-data-analytics.datalake.pgd_tbl_kredit k 
# left join pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch AS vt
#     ON cast(k.branch_code as integer) = cast(sub_branch_cd as integer)
# left join pgd-prod-data-analytics.datalake.gis_master_outlet as g
#     on cast(k.branch_code as integer) = cast(g.kode as integer)
# WHERE 
#   product_code in('01', '02')
#   and trim(client_id) = ''
#   AND k.create_date >= date_sub(current_date(), interval 2 day)
#   AND k.create_date <= date_sub(current_date(), interval 1 day)
#   and vt.sub_branch_nm not like 'BRI%' -- exclude yang colo
#   and branch_code not in('00045', '00050')
#   and cif is not null
#   and cif not like ''
# )
#     select count(*)
#     from ct_row
# """

# total_rows = bq_client.query(count_query).to_dataframe().iloc[0,0]
# print(f"Total rows to fetch: {total_rows}")

# # Set batch size
# batch_size = 500000
# num_batches = math.ceil(total_rows / batch_size)
# print(f"Number of batches: {num_batches}")

# # Fetch data in batches
# all_data = []

# for i in range(num_batches):
#     offset = i * batch_size
#     print(f"Fetching batch {i+1}/{num_batches} (offset: {offset})...")
    
#     batch_query = f"""
#     with ct_row as (
#         SELECT distinct 
#           cif,
#           no_kontrak,
#           up,
#           k.create_date,
#           branch_code,
#           g.latitude,
#           g.longitude
#         FROM
#           pgd-prod-data-analytics.datalake.pgd_tbl_kredit k 
#         left join pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch AS vt
#             ON cast(k.branch_code as integer) = cast(sub_branch_cd as integer)
#         left join pgd-prod-data-analytics.datalake.gis_master_outlet as g
#             on cast(k.branch_code as integer) = cast(g.kode as integer)
#         WHERE
#           trim(client_id) = ''
#           and product_code in('01', '02') -- gadai rahn only
#           AND k.create_date >= date_sub(current_date(), 2)
#           AND k.create_date <= date_sub(current_date(), 1)
#           and vt.sub_branch_nm not like 'BRI%' -- exclude yang colo
#           and branch_code not in('00045', '00050')
#           and cif is not null
#           and cif not like ''
#         )
#     select *
#     from ct_row
#     ORDER BY create_date
#     LIMIT {batch_size} OFFSET {offset}
#     """
    
#     batch_df = bq_client.query(batch_query)
#     all_data.append(batch_df)
#     print(f"  Fetched {len(batch_df)} rows")
    
#     # Break if we got fewer rows than batch_size (last batch)
#     if len(batch_df) < batch_size:
#         break

# # Combine all batches
# try:
#     df = pd.concat(all_data, ignore_index=True)
# except Exception as e:
#     print(f"Error: {e}")

In [ ]:
# !pip install haversine
# import pandas as pd
# from datetime import datetime, timedelta
# from google.api_core.exceptions import NotFound
# from haversine import haversine, Unit

# current_timestamp = pd.Timestamp.now(tz='Asia/Jakarta')

# def check_distance_and_time(df, day, time_limit_minutes, limit_distance):

#   # sort by date
#     df_filtered = df
#     df_filtered['create_date'] = pd.to_datetime(df_filtered['create_date'])
#     df_sorted = df_filtered.sort_values(by='create_date')

#     violating_transactions = []

#     # group by cif
#     for cif, cif_df in df_sorted.groupby('cif'):
#         # Iterate through transactions for each CIF
#         for i in range(len(cif_df)):
#             for j in range(i + 1, len(cif_df)):
#                 txn1 = cif_df.iloc[i]
#                 txn2 = cif_df.iloc[j]

#                 # cek apakah transaksi di beda outlet
#                 if txn1['branch_code'] != txn2['branch_code']:
#                     # cek apakah transaksi berada pada time limit
#                     time_diff = (txn2['create_date'] - txn1['create_date']).total_seconds() / 60
#                     if time_diff <= time_limit_minutes:
#                         # latlong is not null
#                         if pd.notna(txn1['latitude']) and pd.notna(txn1['longitude']) and \
#                            pd.notna(txn2['latitude']) and pd.notna(txn2['longitude']):
#                             # menghitung jarak menggunakan haversine
#                             coords_1 = (txn1['latitude'], txn1['longitude'])
#                             coords_2 = (txn2['latitude'], txn2['longitude'])
#                             distance = haversine(coords_1, coords_2, unit=Unit.KILOMETERS)

#                             # Check if distance is greater than 60 km
#                             if distance / (time_diff + 1) * time_limit_minutes > limit_distance:
                                
#                                 # risk_score = (distance / time_diff) * 60
#                                 # if risk_score <= 100:
#                                 #     risk_level = 'Moderate to High'
#                                 # elif risk_score > 100:
#                                 #     risk_level = 'High'
                                
#                                 # nanti kalo butuh bikin multiple risk 
                                
#                                 # Add both transactions to the violating list with violation value
#                                 txn1_violation = txn1.copy()
#                                 txn1_violation['symptom'] = 'Rentang Jarak Pencairan Gadai Melalui Outlet diluar kewajaran'
#                                 txn1_violation['violation_value'] = distance / (time_diff + 1) * time_limit_minutes
#                                 txn1_violation['violation_value_jarak'] = f'transaksi sejauh {distance:.2f} km dalam waktu {time_diff:.2f} menit'
#                                 violating_transactions.append(txn1_violation)

#                                 txn2_violation = txn2.copy()
#                                 txn2_violation['symptom'] = 'Rentang Jarak Pencairan Gadai Melalui Outlet diluar kewajaran'
#                                 txn2_violation['violation_value'] = distance / (time_diff + 1) * time_limit_minutes
#                                 txn2_violation['violation_value_jarak'] = f'transaksi sejauh {distance:.2f} km dalam waktu {time_diff:.2f} menit'
#                                 violating_transactions.append(txn2_violation)


#     if not violating_transactions:
#         return pd.DataFrame()

#     # Convert the list of violating transactions to a DataFrame and drop duplicates
#     violating_df = pd.DataFrame(violating_transactions).drop_duplicates().reset_index(drop=True)
#     violating_df.drop(columns=['longitude', 'latitude', 'no_kontrak', 'up', 'create_date', 'violation_value'], inplace=True)
#     txn1_violation['violation_value_jarak'] = f'transaksi sejauh {distance:.2f} km dalam rentang {time_diff:.2f} menit'
    
#     violating_df['violation_value_trx'] = f'transaksi sejauh {distance:.2f} km dalam rentang {time_diff:.2f} menit'
#     violating_df['violation_value_gram'] = None
#     violating_df['violation_value_nominal'] = None
#     violating_df['violation_value_cif'] = None
#     violating_df['value_trx'] = None
#     violating_df['value_gram'] = None
#     violating_df['value_nominal'] = None
    
#     violating_df['detection_date'] = current_timestamp.date()
#     violating_df['tingkat_risiko'] = 'High'
#     violating_df['skor_risiko'] = 5
#     violating_df['skor_risiko'] = violating_df['skor_risiko'].astype('object')
    
#     column_order = [
#         'symptom',
#         'cif',
#         'violation_value_trx',
#         'violation_value_nominal',
#         'violation_value_gram',
#         'violation_value_jarak',
#         'violation_value_cif',
#         'value_trx',
#         'value_nominal',
#         'value_gram',
#         'branch_code',
#         'detection_date',
#         'tingkat_risiko',
#         'skor_risiko'
#     ]
    
#     # Only include columns that exist in the DataFrame
#     column_order = [col for col in column_order if col in violating_df.columns]
    
#     # Add any remaining columns not in the specified order
#     remaining_cols = [col for col in violating_df.columns if col not in column_order]
#     final_column_order = column_order + remaining_cols
    
#     violating_df = violating_df[final_column_order]

#     return violating_df

In [ ]:
# try:
#     df_10 = check_distance_and_time(df, 3, 180, 60)
#     print(df_10.head(2)) 
# except Exception as e:
#     print(f"An error occurred: {repr(e)}")

In [ ]:
# try:
#     temp_table = "dev_datamart.tbl_outlier_detection_det_temp"

#     drop_table = f"""
#     DROP TABLE IF EXISTS {temp_table}
#     """
#     sql.execute_ddl(drop_table)


#     create_table = f"""
#     CREATE EXTERNAL TABLE IF NOT EXISTS {temp_table} (
#       symptoms STRING,
#       cif STRING,
#       violation_value_trx STRING,
#       violation_value_nominal STRING,
#       violation_value_gram STRING,
#       violation_value_jarak STRING,
#       violation_value_cif STRING,

#       value_trx string,
#       value_nominal string,
#       value_gram string,


#       branch_code STRING,
#       detection_date DATE,
#       tingkat_risiko STRING,
#       skor_risiko STRING
#     )
#     STORED AS PARQUET
#     """
#     sql.execute_ddl(create_table)

#     df_10 = check_distance_and_time(df, 3, 60, 60)

#     if df_10 is not None and not df_10.empty:
#         print(f"Found {len(df_10)} records to insert")

#         sql.write(db_name = "dev_datamart", 
#                   table_name = "tbl_outlier_detection_det_temp", 
#                   dataframe = df_10, 
#                   operation = "append", 
#                   index_column = 'cif')

#         print(df_10.head(2))

#         s_10 = f"""
#         INSERT INTO {destination_table}
#         SELECT
#           symptoms,
#           cif,
#           violation_value_trx,
#           violation_value_nominal,
#           violation_value_gram,
#           violation_value_jarak,
#           violation_value_cif,

#           null as value_trx,
#           null as value_nominal,
#           null as value_gram,

#           branch_code,
#           DATE_SUB(detection_date, 0) AS detection_date,
#           tingkat_risiko,
#           CAST(skor_risiko AS DOUBLE) AS skor_risiko
#         FROM {temp_table}
#         """
#         sql.execute_ddl(s_10)
#     else:
#         print("df_10 kosong")

#     sql.execute_ddl(drop_table)

# except Exception as e:
#     print(f"An error occurred: {repr(e)}")

##### Symptom 11: Nomor Rekening Bank Pencaiaran Transaksi diluar kewajaran

In [ ]:

# s_8 = f"""
# --INSERT INTO {destination_table}
# WITH base_data AS (
#         SELECT
#             nt.create_date,
#             nt.branch_code as kode_outlet,
#             k.cif,
#             nt.no_rek,
#             vt.sub_branch_nm,
#             vt.branch_nm,
#             vt.area_nm,
#             vt.region_nm
#         FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_nontunai nt
#         LEFT JOIN 
#             pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit k ON 
#                 nt.id_reff = k.no_kontrak 
#                 AND date(nt.create_date) = date(k.tgl_transaksi)
#         LEFT JOIN
#             pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch vt
#             ON CAST(nt.branch_code AS STRING) = CAST(vt.sub_branch_cd AS STRING)
#         LEFT JOIN
#             pgd-prod-data-analytics.datalake.pgd_tbl_transaction_handler th
#             ON nt.tx_code = th.kode_transaksi
#         WHERE
#             nt.no_rek IS NOT NULL
#             AND date(nt.create_date) >= date_sub(current_date(), interval 7 day) 
#             AND date(nt.create_date) <= current_date()
#             AND UPPER(th.description) LIKE '%CAIR%'
#             and nt.product_code in ('01', '02')
#     ),
#     cif_count_per_norek AS (
#         SELECT 
#             no_rek,
#             COUNT(DISTINCT cif) as jumlah_cif_unik
#         FROM base_data
#         WHERE cif IS NOT NULL
#         GROUP BY no_rek
#         HAVING COUNT(DISTINCT cif) > 3
#     )
#     SELECT 
#         --bd.create_date,
#         --bd.no_rek,
#         --c.jumlah_cif_unik as violation_value,
#         'Nomor Rekening Bank Pencairan Transaksi diluar kewajaran' as symptoms,
#         bd.cif,
#         concat(cast(c.jumlah_cif_unik as string), ' CIF menggunakan 1 rekening bank') AS violation_value_trx,
#         null AS violation_value_nominal,
#         null AS violation_value_gram,
#         null AS violation_value_jarak,
#         null AS violation_value_cif,
        
#         null as value_trx,
#         null as value_nominal,
#         null as value_gram,
        
#         bd.kode_outlet as branch_code,
#         current_date() as detection_date,
        
#         case 
#           when jumlah_cif_unik <= 33 then 'Moderate'
#           when jumlah_cif_unik <= 105 then 'Moderate to High'
#           when jumlah_cif_unik > 105 then 'High'
#         end as tingkat_risiko,
#         case
#           when jumlah_cif_unik <= 33 then 3.0
#           when jumlah_cif_unik <= 105 then 4.0
#           when jumlah_cif_unik > 105 then 5.0
#         end as skor_risiko,
#         case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
        
        
#     FROM base_data bd
#     INNER JOIN cif_count_per_norek c 
#         ON bd.no_rek = c.no_rek
# """
# df = bq_client.query(s_8).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

##### Symptom 13: Jumlah Penggunaaan Nomor HP yang Sama di Luar Kewajaran

In [ ]:
# s_13 = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Jumlah Penggunaan Nomor HP yang Sama di Luar Kewajaran' as symptoms,
# base.cif,
# concat('Satu nomor hp digunakan oleh  ', cast(base.jumlah_cif as STRING), ' CIF') as violation_value_trx,
# null as violation_value_nominal,
# null as violation_value_gram,
# null as violation_value_jarak,
# concat('Satu nomor hp digunakan oleh  ', cast(base.jumlah_cif as STRING), ' CIF') as violation_value_cif, --buat symptom ktp, device_id, no hp

# null as value_trx,
# null as value_nominal,
# null as value_gram,

# base.branch_code,
# current_date() AS detection_date,
# case 
#   when base.jumlah_cif = 3 then 'Moderate'
#   when base.jumlah_cif = 4 then 'Moderate to High'
#   when base.jumlah_cif > 4 then 'High'
# end as tingkat_risiko,
# case
#   when base.jumlah_cif = 3 then 3.0
#   when base.jumlah_cif = 4 then 4.0
#   when base.jumlah_cif > 4 then 5.0
# end as skor_risiko,
# base.clean_hp,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
# from(    
#     WITH aktif_cif AS (
#   SELECT 
#       DISTINCT cif,
#       b.region_nm,
#       a.branch_code
#   FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit a
#   JOIN pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch b
#   ON a.branch_code=b.branch_cd
#   WHERE date(tgl_transaksi) >= date_sub(current_date(), interval 7 day)
# ),
# hp_cleaned AS (
#   SELECT
#     c.cif,
#     -- Cleansing: ambil hanya angka (hapus spasi, +62, tanda, dll)
#     --REGEXP_REPLACE(c.no_hp, '[^0-9]', '') AS clean_hp
#     c.telp AS clean_hp,
#     a.region_nm,
#     a.branch_code
#   FROM pgd-prod-data-analytics.datalake.pgd_tbl_customer c
#   JOIN aktif_cif a
#     ON c.cif = a.cif
#   WHERE c.telp IS NOT NULL
#     AND c.is_active = '1'
# ),
# hp_group AS (
#   SELECT
#     clean_hp,
#     COUNT(DISTINCT cif) AS jumlah_cif,
#     region_nm
#   FROM hp_cleaned h
#   GROUP BY clean_hp, region_nm
#   HAVING COUNT(DISTINCT cif) > 2
# )
# SELECT
#   g.clean_hp,
#   g.jumlah_cif,
#   h.cif,
#   h.region_nm,
#   h.branch_code
# FROM hp_group g
# JOIN hp_cleaned h
#   ON g.clean_hp = h.clean_hp
# --ORDER BY g.jumlah_cif DESC, g.clean_hp, h.region_nm
# )
#     AS base
# """
# df_s_13 = bq_client.query(s_13).to_dataframe()
# df_s_13 = df_s_13.sort_values(by='violation_value_cif', ascending=True)
# df_s_13 = df_s_13[df_s_13['clean_hp'].str.startswith('8') |
#            df_s_13['clean_hp'].str.startswith('2') |
#            df_s_13['clean_hp'].str.startswith('3') |
#            df_s_13['clean_hp'].str.startswith('4') |
#            df_s_13['clean_hp'].str.startswith('08')| 
#            df_s_13['clean_hp'].str.startswith('02')| 
#            df_s_13['clean_hp'].str.startswith('03')| 
#            df_s_13['clean_hp'].str.startswith('04')| 
#            df_s_13['clean_hp'].str.startswith('08') & 
#            ~df_s_13['clean_hp'].str.startswith('00')]
# df_s_13=df_s_13.drop('clean_hp', axis=1)
# df_s_13 = df_s_13.drop_duplicates()
# df_s_13['detection_date'] = pd.to_datetime(df_s_13['detection_date'], errors='coerce')
# df_s_13['skor_risiko'] = df_s_13['skor_risiko'].astype(float)
# df=df_s_13
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

##### Symptom 14: Transaksi Anomali Transfer Tabungan Emas

###### Many-to-One Transfer TE dalam 1 hari

In [ ]:


# from datetime import datetime, timedelta, date
# from dateutil.relativedelta import relativedelta

# today = datetime.today()
# yesterday = today - relativedelta(days=1)

# today=today.strftime('%Y-%m-%d')
# yesterday=yesterday.strftime('%Y-%m-%d')

# print(today)

# s_14a = f"""
# with transaksi as 
#     (
#     select
#         t.norek,
#         t.cif,
#         t.customer_name,
#         t.create_date,
#         t.tgl_transaksi,
#             case 
#                 when upper(t.jenis_transaksi) like 'BUY' then 'Buyback'
#                 when upper(t.jenis_transaksi) like 'SALE' then 'Buy'
#                 when upper(t.jenis_transaksi) like 'TRANSFER' then 'Transfer'
#                 when upper(t.jenis_transaksi) like 'TRANSFERKS' then 'Transfer'
#                 when upper(t.jenis_transaksi) like 'OPEN' then 'Open'
#                 else '' end as 
#         jenis_transaksi,
#         t.hutang_transaksi AS nominal_trx,
#         det.berat,
#         t.norek_kredit norek_penerima,
#         tab.cif cif_penerima,
#         tab.customer_name customer_name_penerima

#      FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_tabungan t
#         LEFT JOIN pgd-prod-data-analytics.datalake.pgd_tbl_e_transaksi_emas_det det 
#             ON t.id = det.id
#         left join (select distinct cif, norek, customer_name from pgd-prod-data-analytics.datalake.pgd_tbl_tabungan) tab
#             on t.norek_kredit = tab.norek
            
#         WHERE 
#             --t.jenis_transaksi IN ('OPEN', 'BUY', 'SALE', 'TRANSFER', 'TRANSFERKS')
#             t.jenis_transaksi IN ('BUY', 'SALE', 'TRANSFER', 'TRANSFERKS')
#             AND date(create_date) >= date_sub(current_date(),interval 1 day)
#             and date(create_date) <= date_sub(current_date(),interval 1 day)

#     ),
# suspect_cif as 
#     (
#     select  
#         tgl_transaksi,
#         count(distinct cif)     as ct_cif_pengirim,
#         cif_penerima,
#         round(sum(berat),4)     as akumulasi_gram,
#         count(cif)              as jumlah_transaksi,
#         jenis_transaksi,
#         current_date()          as as_dt
#     from transaksi
#     where jenis_transaksi like 'Transfer' and cif != cif_penerima
#     group by tgl_transaksi, cif_penerima, jenis_transaksi, as_dt
#     having count(distinct cif) >= 5 and sum(berat) >= 5 ----------------- rules were here
#     )
# select distinct
#     'Anomali TE: Many-to-One Transfer' as symptoms, 
#     sc.cif_penerima as cif,
#     concat('Mendapat transfer TE dari ', cast(sc.ct_cif_pengirim as string), ' CIF berbeda, dengan total ', cast(round(akumulasi_gram,4) as string), ' gram') as violation_value_trx,
#     cast(null as string) as violation_value_nominal,
#     cast(null as string) as violation_value_gram,
#     cast(null as string) as violation_value_jarak,
#     cast(null as string) as violation_value_cif,
#     cast(null as string) as branch_code,
#     CAST(current_date() AS timestamp) AS detection_date,
#     case
#         when sc.ct_cif_pengirim >= 15 then 'High' -----------------------rules were here
#         when sc.ct_cif_pengirim >= 10 then 'Moderate to High' -----------rules were here
#         else 'Moderate'
#         end as
#     tingkat_risiko,
#     case
#         when sc.ct_cif_pengirim >= 15 then 5.0 --------------------------rules were here
#         when sc.ct_cif_pengirim >= 10 then 4.0 --------------------------rules were here
#         else 3.0
#         end as
#     skor_risiko,

#     cast(null as float64) as value_nominal,
#     cast (sc.ct_cif_pengirim as float64) as value_trx,
#     round(
#       cast(sc.akumulasi_gram as float64), 4) as value_gram,
#     case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source
       

# from suspect_cif as sc
#     inner join transaksi t
#         on sc.cif_penerima = t.cif_penerima
# """

# df = bq_client.query(s_14a).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)


###### One-to-Many Transfer TE dalam 1 hari

In [ ]:

# from datetime import datetime, timedelta, date
# from dateutil.relativedelta import relativedelta

# today = datetime.today()
# yesterday = today - relativedelta(days=1)

# today=today.strftime('%Y-%m-%d')
# yesterday=yesterday.strftime('%Y-%m-%d')

# print(today)

# s_14b = f"""
#     with transaksi as 
#         (
#         select
#             t.norek,
#             t.cif,
#             t.customer_name,
#             t.create_date,
#             t.tgl_transaksi,
#                 case 
#                     when upper(t.jenis_transaksi) like 'BUY' then 'Buyback'
#                     when upper(t.jenis_transaksi) like 'SALE' then 'Buy'
#                     when upper(t.jenis_transaksi) like 'TRANSFER' then 'Transfer'
#                     when upper(t.jenis_transaksi) like 'TRANSFERKS' then 'Transfer'
#                     when upper(t.jenis_transaksi) like 'OPEN' then 'Open'
#                     else '' end as 
#             jenis_transaksi,
#             t.hutang_transaksi AS nominal_trx,
#             det.berat,
#             t.norek_kredit norek_penerima,
#             tab.cif cif_penerima,
#             tab.customer_name customer_name_penerima

#          FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_tabungan t
#             LEFT JOIN pgd-prod-data-analytics.datalake.pgd_tbl_e_transaksi_emas_det det 
#                 ON t.id = det.id
#             left join (select distinct cif, norek, customer_name from pgd-prod-data-analytics.datalake.pgd_tbl_tabungan) tab
#                 on t.norek_kredit = tab.norek

#             WHERE 
#                 --t.jenis_transaksi IN ('OPEN', 'BUY', 'SALE', 'TRANSFER', 'TRANSFERKS')
#                 t.jenis_transaksi IN ('BUY', 'SALE', 'TRANSFER', 'TRANSFERKS')
#                 AND date(create_date) >= date_sub(current_date(), interval 1 day)
#                 and date(create_date) <= date_sub(current_date(), interval 1 day)

#         ),
#     suspect_cif as 
#         (
#         select  
#             tgl_transaksi,
#             cif                    as cif_pengirim,
#             count(distinct cif_penerima) as ct_cif_penerima,
#             sum(berat)              as akumulasi_gram,
#             count(cif)              as jumlah_transaksi,
#             jenis_transaksi,
#             current_date()          as as_dt
#         from transaksi
#         where jenis_transaksi like 'Transfer' and cif != cif_penerima
#         group by tgl_transaksi, cif_pengirim, jenis_transaksi, as_dt
#         having count(distinct cif_penerima) >= 2 and sum(berat) >= 5 ----------------- rules were here
#         )

#     select distinct
#         'Anomali TE: One-to-Many Transfer' as symptoms, 
#         sc.cif_pengirim as cif,
#         concat('Melakukan transfer TE ke ', cast(sc.ct_cif_penerima as string), ' CIF berbeda, dengan total ', cast(akumulasi_gram as string), ' gram') as violation_value_trx,
#         cast(null as string) as violation_value_nominal,
#         cast(null as string) as violation_value_gram,
#         cast(null as string) as violation_value_jarak,
#         cast(null as string) as violation_value_cif,
#         cast(null as string) as branch_code,
#         CAST(current_date() AS timestamp) AS detection_date,
#         case
#             when sc.ct_cif_penerima >= 10 then 'High' -----------------------rules were here
#             when sc.ct_cif_penerima >= 5 then 'Moderate to High' -----------rules were here
#             else 'Moderate'
#             end as
#         tingkat_risiko,
#         case
#             when sc.ct_cif_penerima >= 10 then 5.0 --------------------------rules were here
#             when sc.ct_cif_penerima >= 5 then 4.0 --------------------------rules were here
#             else 3.0
#             end as
#         skor_risiko,
#         cast (sc.ct_cif_penerima as float64) as value_trx,
#         cast (null as float64) as value_nominal,
#         round(
#             cast (sc.akumulasi_gram as float64),4) as value_gram,
#         case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source



#     from suspect_cif as sc
#         inner join transaksi t
#             on sc.cif_pengirim = t.cif
#     """
# df=[]
# df = bq_client.query(s_14b).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)


###### Anomali TE: Frekuensi dan Volume Transaksi Mencurigakan

In [ ]:


# from datetime import datetime, timedelta, date
# from dateutil.relativedelta import relativedelta

# today = datetime.today()
# yesterday = today - relativedelta(days=1)

# today=today.strftime('%Y-%m-%d')
# yesterday=yesterday.strftime('%Y-%m-%d')


# s_14c = f"""
# --INSERT INTO {destination_table}
# SELECT
# 'Anomali TE: Frekuensi dan Volume Transaksi Mencurigakan' as symptoms,
# cif,
# CAST(NULL AS STRING) as violation_value_trx,
# concat('Transaksi ', cast(frequency as string), ' kali, dengan volume sebesar ', cast(volume_gold_gram as string), ' gram, dalam 3 hari terakhir')  as violation_value_nominal,
# CAST(NULL AS STRING) as violation_value_gram,
# CAST(NULL AS STRING) AS violation_value_jarak,
# CAST(NULL AS STRING) AS violation_value_cif,

# CAST(frequency AS string) as value_trx,
# cast(null as string) as value_nominal, 
# cast(ROUND(CAST(volume_gold_gram AS FLOAT64), 4) as string) as value_gram,

# branch_code,
# CAST(current_date() AS DATE) AS detection_date,
# case 
#   when volume_gold_gram <= 75 then 'Moderate'
#   when volume_gold_gram <= 125 then 'Moderate to High'
#   when volume_gold_gram > 125 then 'High'
# end as tingkat_risiko,
# case
#   when volume_gold_gram <= 75 then 3.0
#   when volume_gold_gram <= 125 then 4.0
#   when volume_gold_gram > 125 then 5.0
# end as skor_risiko,
# case
#     when symptoms LIKE '%Machine Learning%' then 'Machine Learning'
#     else 'Rule-Based'
# end as alert_source


# from(    
#     with transaksi_base as (
#     select
#         t.cif as cif_pengirim,
#         tab.cif as cif_penerima,
#         case 
#             when upper(t.jenis_transaksi) like 'BUY' then 'Buyback'
#             when upper(t.jenis_transaksi) like 'SALE' then 'Buy'
#             when upper(t.jenis_transaksi) like 'TRANSFER' then 'Transfer'
#             else 'Other' 
#         end as jenis_transaksi,
#         t.create_date,
#         t.hutang_transaksi as nominal_trx,
#         det.berat
#     FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_tabungan t
#     LEFT JOIN pgd-prod-data-analytics.datalake.pgd_tbl_e_transaksi_emas_det det ON t.id = det.id
#     LEFT JOIN (select distinct cif, norek from pgd-prod-data-analytics.datalake.pgd_tbl_tabungan) tab ON t.norek_kredit = tab.norek
#     WHERE t.jenis_transaksi IN ('BUY', 'SALE', 'TRANSFER', 'TRANSFERKS')
#       -- Filter for the last 7 weeks
#       AND date(t.create_date) >= date_sub(current_date(), interval 3 day)
#       AND date(t.create_date) <= date_sub(current_date(), interval 1 day)
# ),
# unpivoted_data as (
#     -- Perspective: SENDER / ACCOUNT HOLDER
#     select 
#         cif_pengirim as cif,
#         jenis_transaksi,
#         case 
#             when jenis_transaksi = 'Buy' then 'Trx In'      -- Gold In
#             when jenis_transaksi = 'Buyback' then 'Trx Out'  -- Gold Out
#             when jenis_transaksi = 'Transfer' then 'Trx Out' -- Gold Out
#             else 'Out' 
#         end as type_transaksi,
#         create_date,
#         berat
#     from transaksi_base
#     where cif_pengirim is not null

#     UNION ALL

#     -- Perspective: RECEIVER (Specific to Transfers)
#     select 
#         cif_penerima as cif,
#         jenis_transaksi,
#         'Trx In' as type_transaksi, -- Receiving a transfer is Gold In
#         create_date,
#         berat
#     from transaksi_base
#     where cif_penerima is not null and jenis_transaksi = 'Transfer'
# )
# select 
#     ud.cif,
#     c.branch_code,
#     --current_date() as start_of_week,
#     --type_transaksi,
#     count(*) as frequency,
#     sum(ud.berat) as volume_gold_gram
# from unpivoted_data ud
# left join pgd-prod-data-analytics.datalake.pgd_tbl_customer c
#   on ud.cif = c.cif
# group by cif, branch_code
# having count(*) >= 15 and sum(berat) >= 50
#     ) x
# """

# df = bq_client.query(s_14c).to_dataframe()
# print('total rows: ', len(df))
# print(df.head(2))
# import_detail_symptom(df,destination_table)

##### Symptom 15: Transaksi dan Biodata Anomali Representasi Graf oleh Machine Learning 

In [38]:
s_15 = f"""
select
ml.cif,
case
    when (ml.connection_proba_norm > 0.5) and (ml.transaction_proba_norm > 0.5) then 'Transaksi Anomali oleh Machine Learning, Koneksi Identitas Anomali oleh Machine Learning'
    when ml.connection_proba_norm > 0.5 then 'Koneksi Identitas Anomali oleh Machine Learning'
    when ml.transaction_proba_norm > 0.5 then 'Transaksi Anomali oleh Machine Learning'
end as symptoms,
CAST(NULL AS STRING) as violation_value_trx,
CAST(NULL AS STRING)  as violation_value_nominal,
CAST(NULL AS STRING) as violation_value_gram,
CAST(NULL AS STRING) AS violation_value_jarak,
CAST(NULL AS STRING) AS violation_value_cif,
CAST(NULL AS FLOAT64) as value_trx,
cast(null as FLOAT64) as value_nominal, 
CAST(NULL AS FLOAT64) as value_gram,
cs.branch_code,
CAST(current_date AS DATE) AS detection_date,
case 
  when (ml.combined_proba >= 0.5) and (ml.combined_proba < 0.7) then 'Moderate'
  when (ml.combined_proba >= 0.7) and (ml.combined_proba < 0.9) then 'Moderate to High'
  when (ml.combined_proba >= 0.9) and (ml.combined_proba < 1.0) then 'High'
end as tingkat_risiko,
case
  when (ml.combined_proba >= 0.5) and (ml.combined_proba < 0.7) then 3.0
  when (ml.combined_proba >= 0.7) and (ml.combined_proba < 0.9) then 4.0
  when (ml.combined_proba >= 0.9) and (ml.combined_proba < 1.0) then 5.0
end as skor_risiko,
'Machine Learning' as alert_source

from pgd-dev-data-analytics.datamart_audit.dm_dt_sidik_ml_customer ml
left join pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_customer cs
on ml.cif = cs.cif
where current_date = current_date()

"""
df = bq_client.query(s_15).to_dataframe()
print('total rows: ', len(df))
print(df.head(2))
import_sum_symptom(df,destination_table)

total rows:  1214177
          cif                                           symptoms  \
0  9005500899  Transaksi Anomali oleh Machine Learning, Konek...   
1  9005500899  Transaksi Anomali oleh Machine Learning, Konek...   

  violation_value_trx violation_value_nominal violation_value_gram  \
0                None                    None                 None   
1                None                    None                 None   

  violation_value_jarak violation_value_cif  value_trx  value_nominal  \
0                  None                None        NaN            NaN   
1                  None                None        NaN            NaN   

   value_gram branch_code detection_date tingkat_risiko  skor_risiko  \
0         NaN       10003     2026-04-16       Moderate          3.0   
1         NaN       10003     2026-04-16       Moderate          3.0   

       alert_source  
0  Machine Learning  
1  Machine Learning  
loaded 1214177 rows
 to pgd-dev-data-analytics.datamart_audi

### Tabel untuk get_suspicious_customers
### dm_tbl_sidik_sum_suspicious_cust

In [ ]:
# sama dg diatas dengan kolom: 
            # a.cif,
            # a.symptoms,
            # a.detection_date,
            # a.tingkat_risiko,
            # b.branch_nm,
            # b.sub_branch_nm 
# karena sudah dilakukan summary table

In [39]:
df.head()

,cif,symptoms,violation_value_trx,violation_value_nominal,violation_value_gram,violation_value_jarak,violation_value_cif,value_trx,value_nominal,value_gram,branch_code,detection_date,tingkat_risiko,skor_risiko,alert_source
0,9005500899,"Transaksi Anomali oleh Machine Learning, Konek...",None,None,None,None,None,NaN,NaN,NaN,10003,2026-04-16,Moderate,3.0,Machine Learning
1,9005500899,"Transaksi Anomali oleh Machine Learning, Konek...",None,None,None,None,None,NaN,NaN,NaN,10003,2026-04-16,Moderate,3.0,Machine Learning
2,9005500899,"Transaksi Anomali oleh Machine Learning, Konek...",None,None,None,None,None,NaN,NaN,NaN,10003,2026-04-16,Moderate,3.0,Machine Learning
3,9005533477,"Transaksi Anomali oleh Machine Learning, Konek...",None,None,None,None,None,NaN,NaN,NaN,10017,2026-04-16,Moderate,3.0,Machine Learning
4,9005678713,"Transaksi Anomali oleh Machine Learning, Konek...",None,None,None,None,None,NaN,NaN,NaN,10081,2026-04-16,Moderate,3.0,Machine Learning


In [40]:
source_table = 'pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_det_suspicious_cust'
sum_table = f"""
select
base.detection_date,
base.cif,
base.jumlah_symptoms,
base.symptoms,
c.branch_code as kode_outlet,
vt.sub_branch_nm as nama_outlet,
vt.branch_nm as nama_cabang,
case
    when average_risiko <= 3.5 then 'Moderate'
    when average_risiko <= 4.5 then 'Moderate to High'
    when average_risiko > 4.5 then 'High'
end as tingkat_risiko, 
alert_source
from (
    SELECT 
    cif,
    string_agg(distinct symptoms, ', ') AS symptoms,
    string_agg(distinct alert_source, ', ') AS alert_source,
    cast(count(distinct symptoms) as integer) as jumlah_symptoms,
    avg(skor_risiko) as average_risiko,
    detection_date
    FROM (
        select distinct *
        from
        {destination_table}
        where skor_risiko is not null
        ) d
    where date(detection_date) = date_sub(current_date(), interval 1 day)
    group by cif, detection_date) base
left join datalake.pgd_tbl_customer c
    on base.cif = c.cif
left join pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch vt
    on c.branch_code = vt.sub_branch_cd
order by base.jumlah_symptoms desc

"""
df_sum = bq_client.query(sum_table).to_dataframe()
print('total rows: ', len(df_sum))
print(df_sum.head(2))


total rows:  21537
             detection_date         cif  jumlah_symptoms  \
0 2026-04-14 00:00:00+00:00  9005850361                4   
1 2026-04-14 00:00:00+00:00  1001684989                4   

                                            symptoms kode_outlet  \
0  Jumlah Rekening/Kontrak Kredit dalam Sehari, N...       10154   
1  Jumlah Rekening/Kontrak Kredit dalam Sehari, N...       10154   

     nama_outlet    nama_cabang tingkat_risiko alert_source  
0  CP SIDIKALANG  CP SIDIKALANG       Moderate   Rule-Based  
1  CP SIDIKALANG  CP SIDIKALANG       Moderate   Rule-Based  


In [41]:
df_sum.columns

Index(['detection_date', 'cif', 'jumlah_symptoms', 'symptoms', 'kode_outlet',
       'nama_outlet', 'nama_cabang', 'tingkat_risiko', 'alert_source'],
      dtype='object')

In [42]:
import_sum_symptom(df_sum,summary_table)

loaded 21537 rows
 to pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_sum_suspicious_cust


### Tabel untuk fetch_caseid
### dm_tbl_sidik_case_suspicious_cust

In [43]:

c_table = f"""

WITH latest_table_case AS (
    -- Ngambil yang masuk di case_id cif-nya apa aja buat digabungin dengan table_summary
    SELECT 
        case_id,
        detection_date,
        cif,
        symptoms,
        status,
        ROW_NUMBER() OVER (PARTITION BY cif ORDER BY detection_date DESC) AS rn
    FROM pgd-dev-data-analytics.datamart_audit.dm_det_tbl_fds_suspicious_case ----ganti saat sudah tak terpakai
    WHERE date(detection_date) >= date_sub(current_date(), interval 30 day)
),
ranked_table_summary AS (
    -- ngambil data dari summary
    SELECT 
        date(detection_date) detection_date,
        cif,
        jumlah_symptoms,
        symptoms,
        alert_source,
        tingkat_risiko,
        LAG(symptoms) OVER (PARTITION BY cif ORDER BY detection_date) AS prev_symptoms_sum,
        LAG(jumlah_symptoms) OVER (PARTITION BY cif ORDER BY detection_date) AS prev_jumlah_symptoms_sum,
        LAG(detection_date) OVER (PARTITION BY cif ORDER BY detection_date) AS prev_detection_date_sum
    FROM {summary_table}
    where date(detection_date) >= date_sub(current_date(), interval 1 day) 
),
combined_history AS (
    -- Gabungan dari Table Summary & CaseID -- tambahin jumlah symptom buat yang case-id
    SELECT 
        cif,
        date(detection_date) detection_date,
        symptoms,
        CASE 
            WHEN LENGTH(symptoms) - LENGTH(REPLACE(symptoms, ',', '')) + 1 > 0 
            THEN LENGTH(symptoms) - LENGTH(REPLACE(symptoms, ',', '')) + 1
            ELSE 1 
        END AS jumlah_symptoms,
        'case' AS source
    FROM pgd-dev-data-analytics.datamart_audit.dm_det_tbl_fds_suspicious_case ----ganti saat sudah tak terpakai
    
    UNION ALL
    
    SELECT 
        cif,
        date(detection_date) detection_date,
        symptoms,
        jumlah_symptoms,
        'summary' AS source
    FROM {summary_table}
),
latest_per_cif AS (
    -- Ambil record terakhir per CIF dari gabungan table A dan B
    SELECT 
        cif,
        date(detection_date) AS last_detection_date,
        symptoms AS last_symptoms,
        jumlah_symptoms AS last_jumlah_symptoms,
        ROW_NUMBER() OVER (PARTITION BY cif ORDER BY detection_date DESC) AS rn
    FROM combined_history
),
filtered_table_summary AS (
    -- Filter data dari table B berdasarkan rules
    SELECT 
        rb.detection_date,
        rb.cif,
        rb.jumlah_symptoms,
        rb.symptoms,
        rb.alert_source,
        rb.tingkat_risiko,
        lpc.last_detection_date,
        lpc.last_symptoms,
        lpc.last_jumlah_symptoms
    FROM ranked_table_summary rb
    LEFT JOIN latest_per_cif lpc ON rb.cif = lpc.cif AND lpc.rn = 1
    WHERE 
        -- Record pertama untuk CIF yang belum ada 
        lpc.cif IS NULL
        OR
        -- Symptoms berbeda DAN tidak dalam 30 hari terakhir
        (rb.symptoms != lpc.last_symptoms AND date_diff(date(rb.detection_date), date(lpc.last_detection_date), day) > 30)
        OR
        -- Symptoms berbeda DAN jumlah symptoms bertambah
        (rb.symptoms != lpc.last_symptoms AND rb.jumlah_symptoms > lpc.last_jumlah_symptoms)
),
new_cases AS (
    -- Generate case_id baru untuk data yang perlu ditambahkan
    SELECT 
        CONCAT(cif, cast(detection_date as string), cast(jumlah_symptoms as string)) AS case_id,
        detection_date,
        cif,
        symptoms,
        alert_source,
        null AS status,
        tingkat_risiko
    FROM filtered_table_summary
)
SELECT 
    case_id,
    detection_date,
    cif,
    symptoms,
    alert_source,
    cast(status as string) status,
    cast(NULL as string) assignee,
    cast(NULL as string) keterangan,
    cast(current_date() as timestamp) update_time
FROM new_cases
WHERE case_id NOT IN (
SELECT case_id FROM pgd-dev-data-analytics.datamart_audit.dm_det_tbl_fds_suspicious_case ----ganti saat sudah tak terpakai 
where date(detection_date) >= date_sub(current_date(), interval 30 day)) 
ORDER BY detection_date desc;
"""

df_case = bq_client.query(c_table).to_dataframe()
print('total rows: ', len(df_case))
print(df_case.head(2))
import_sum_symptom(df_case,case_table)


total rows:  0
Empty DataFrame
Columns: [case_id, detection_date, cif, symptoms, alert_source, status, assignee, keterangan, update_time]
Index: []
DataFrame is empty. Skipping upload to pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_case_suspicious_cust



### Untuk tabel fetch suspicious customers

In [44]:
fetch_sus = f"""
with base as(
        SELECT
            a.cif,
            a.detection_date,
            string_agg(distinct s.violation_value_trx, ', ') AS violation_value_trx,
            string_agg(distinct s.violation_value_nominal, ', ') as violation_value_nominal,
            string_agg(distinct s.violation_value_gram, ', ') as violation_value_gram,
            string_agg(distinct s.violation_value_jarak, ', ') as violation_value_jarak,
            string_agg(distinct s.violation_value_cif, ', ') as violation_value_cif,
        FROM
            pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_sum_suspicious_cust a
        LEFT JOIN pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_det_suspicious_cust s
            ON a.cif=s.cif and a.detection_date=s.detection_date
        GROUP BY
            a.cif, a.detection_date)
        
        SELECT DISTINCT
            a.cif,
            a.symptoms,
            base.violation_value_trx,
            s.value_trx,
            base.violation_value_nominal,
            s.value_nominal,
            base.violation_value_gram,
            s.value_gram,
            base.violation_value_jarak,
            base.violation_value_cif,
            a.kode_outlet as branch_code,
            a.detection_date,
            b.branch_nm,
            b.area_nm,
            b.region_nm,
            b.sub_branch_nm AS nama_outlet,
            a.tingkat_risiko,
            e.nik_pendek AS nik_karyawan,
            CASE
                WHEN a.symptoms = 'Koneksi Identitas Anomali oleh Machine Learning' THEN 'Machine Learning'
                WHEN a.symptoms = 'Transaksi Anomali oleh Machine Learning' THEN 'Machine Learning'
                ELSE 'Rule-Based'
            END as alert_source
        FROM pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_sum_suspicious_cust a
        LEFT JOIN pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_det_suspicious_cust s
          ON a.cif=s.cif and a.detection_date=s.detection_date
        left join base as base
          on a.cif=base.cif and a.detection_date=base.detection_date
        LEFT JOIN pgd-dev-data-analytics.datamart_audit.dm_det_vt_lookup_branch b
          ON a.kode_outlet=b.sub_branch_cd
        LEFT JOIN pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_customer d
          ON a.cif=d.cif
        left join pgd-dev-data-analytics.datalake.hcms_emp_cur_data e --kalau sudah ada aralia_public_ms_employee, pakai.
          on d.ktp=e.ktp
        where b.sub_branch_cd is not null and b.sub_branch_nm is not null and a.kode_outlet is not null
        --and a.detection_date = '{today} 00:00:00 UTC'
        ORDER BY detection_date DESC

"""

df_sus = bq_client.query(fetch_sus).to_dataframe()
print('total rows: ', len(df_sus))
print(df_sus.head(2))
import_sum_symptom(df_sus, "pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_fetch_suspicious_cust")

total rows:  3330977
          cif                                           symptoms  \
0  1032758467  Nomor Rekening Bank Pencairan Transaksi diluar...   
1  9000151431  Nomor Rekening Bank Pencairan Transaksi diluar...   

                  violation_value_trx  value_trx violation_value_nominal  \
0  29 CIF menggunakan 1 rekening bank        NaN                    <NA>   
1   5 CIF menggunakan 1 rekening bank        NaN                    <NA>   

   value_nominal violation_value_gram  value_gram violation_value_jarak  \
0            NaN                 <NA>         NaN                  <NA>   
1            NaN                 <NA>         NaN                  <NA>   

  violation_value_cif branch_code            detection_date       branch_nm  \
0                <NA>       13259 2026-04-14 00:00:00+00:00  CP UJUNGBERUNG   
1                <NA>       13462 2026-04-14 00:00:00+00:00     CP PEMALANG   

          area_nm        region_nm     nama_outlet tingkat_risiko  \
0  AREA BAND

### UNtuk fetch caseid customer

In [45]:
fetch_case = f"""
SELECT DISTINCT
            ca.case_id,
            ca.cif,
            ca.detection_date,
            ca.symptoms,
            su.tingkat_risiko,
            CASE
            WHEN ca.status IS NULL THEN 'null'
            ELSE ca.status
            END as status,
            ca.keterangan
        FROM `pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_case_suspicious_cust` ca
        LEFT JOIN `pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_sum_suspicious_cust` su
            ON ca.cif = su.cif AND ca.detection_date = su.detection_date AND ca.symptoms = su.symptoms
        LEFT JOIN `pgd-dev-data-analytics.datamart_audit.dm_det_vt_lookup_branch` vt
            ON su.kode_outlet = vt.sub_branch_cd
        --WHERE ca.detection_date = '{today} 00:00:00 UTC'
        ORDER BY ca.detection_date DESC, su.tingkat_risiko DESC
"""

df_fetch_case = bq_client.query(fetch_case).to_dataframe()
print('total rows: ', len(df_fetch_case))
print(df_fetch_case.head(2))
import_sum_symptom(df_fetch_case, "pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_fetch_caseid_cust")

total rows:  12437
                 case_id         cif            detection_date  \
0  10221567402026-04-132  1022156740 2026-04-13 00:00:00+00:00   
1  90018715782026-04-132  9001871578 2026-04-13 00:00:00+00:00   

                                            symptoms    tingkat_risiko status  \
0  Jumlah Rekening/Kontrak Kredit dalam Seminggu,...  Moderate to High   null   
1  Nomor Rekening Bank Pencairan Transaksi diluar...  Moderate to High   null   

  keterangan  
0       None  
1       None  
loaded 12437 rows
 to pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_fetch_caseid_cust
